<a href="https://colab.research.google.com/github/ahamedammaar29-star/Megumi-Utsukuhi/blob/main/Megumi_ASI_Master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌸 Megumi Utsukuhi — Master ASI Project

**Gemini 3.8 Flash · Persistent second memory · RAG knowledge base · Agent workflow · Reports · Full-screen 3D VRM avatar · Multilingual TTS**

This is a clean replacement for the previous overlapping notebook. It separates four layers: (1) Gemini reasoning, (2) explicit long-term memories, (3) a searchable document knowledge base, and (4) the avatar/voice interface. It uses the supplied Maki Hiroshi VRM model as Megumi's current 3D visual.

> Run the cells from top to bottom. Add a **newly rotated** key in Colab Secrets before running the Gemini initialization cell. The exposed key visible in the image you shared must not be reused.

## Quick commands inside Megumi's keyboard

- `/remember <fact>` — save an explicitly approved fact to long-term memory.
- `/recall <query>` — retrieve relevant memories/documents.
- `/forget-source <exact source>` — remove chunks from a named source.
- `/memory-status` — inspect memory counts.
- `/deep <question>` — planner → specialist → reviewer pipeline.
- `/report <topic>` — generate a Markdown report and save it to Google Drive.
- `/clear` — clear current and persisted chat history; saved memories/documents remain.

The agent uses Google Search grounding for queries that look time-sensitive. Documents are indexed only when you explicitly upload them or run the folder-ingestion cell. Memory/RAG is retrieval, not model retraining or consciousness.

In [ ]:
# @title 1. FIXED — Install dependencies, mount Drive, and load Master ZIP
!pip -q install -U "google-genai>=1.53.0" "chromadb>=1.0.20" "sentence-transformers>=5.1.0" "pypdf>=5.0.0" "python-docx>=1.1.0" "beautifulsoup4>=4.12.0"

from pathlib import Path
import os, sys, zipfile, shutil
from google.colab import drive, files

# Added force_remount=True to resolve temporary mounting conflicts
drive.mount('/content/drive', force_remount=True)

# Detect an already extracted project first
PROJECT_ROOT = Path('/content/Megumi_ASI_Master')

if not (PROJECT_ROOT / 'megumi_core.py').exists():
    # Look for an already uploaded ZIP
    zip_files = list(Path('/content').glob('*.zip'))
    if not zip_files:
        print("Please upload Megumi_ASI_Master.zip")
        uploaded = files.upload()
        zip_files = [
            Path('/content') / name
            for name in uploaded
            if name.lower().endswith('.zip')
        ]

    if not zip_files:
        raise RuntimeError("Master ZIP not found.")

    archive_path = zip_files[0]
    print("Extracting:", archive_path.name)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall('/content')

# Correct folder name from the actual ZIP
PROJECT_ROOT = Path('/content/Megumi_ASI_Master')

if not (PROJECT_ROOT / 'megumi_core.py').exists():
    raise FileNotFoundError(
        "megumi_core.py is still missing. "
        "Please send me the ZIP contents or a screenshot."
    )

sys.path.insert(0, str(PROJECT_ROOT))

# Persistent Google Drive storage
BASE_DIR = Path('/content/drive/MyDrive/MegumiASI')
MEMORY_DIR = BASE_DIR / 'memory' / 'chroma_db'
KNOWLEDGE_DIR = BASE_DIR / 'knowledge'
REPORTS_DIR = BASE_DIR / 'reports'
AUDIO_DIR = BASE_DIR / 'audio'
ASSETS_DIR = BASE_DIR / 'assets'

for folder in (MEMORY_DIR, KNOWLEDGE_DIR, REPORTS_DIR, AUDIO_DIR, ASSETS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Copy avatar into persistent Drive storage
PACKAGE_AVATAR = PROJECT_ROOT / 'assets' / 'Maki_Hiroshi.vrm'
AVATAR_PATH = ASSETS_DIR / 'Maki_Hiroshi.vrm'

if PACKAGE_AVATAR.exists() and not AVATAR_PATH.exists():
    shutil.copy2(PACKAGE_AVATAR, AVATAR_PATH)

print("\n===================================")
print("MEGUMI ASI PROJECT LOADED")
print("===================================")
print("Project:", PROJECT_ROOT)
print("Core:", (PROJECT_ROOT / 'megumi_core.py').exists())
print("Avatar:", AVATAR_PATH.exists())
print("Memory:", MEMORY_DIR)
print("===================================")

Mounted at /content/drive
Please upload Megumi_ASI_Master.zip


In [ ]:
# @title 2. Secure Gemini 3.8 Flash brain initialization and real API test
from google import genai
from google.colab import userdata

MODEL_NAME = 'gemini-3.8-flash'
TTS_MODEL_NAME = 'gemini-3.8-flash-tts'

# First revoke the key exposed in the photo. Add a NEW key in Colab → Secrets:
# Name = GEMINI_API_KEY; Value = newly generated key; enable notebook access.
_api_key = userdata.get('GEMINI_API_KEY')
if not _api_key:
    raise RuntimeError('GEMINI_API_KEY missing. Add it in Colab → Secrets and enable notebook access.')
GEMINI_CLIENT = genai.Client(api_key=_api_key)
del _api_key  # Keep the raw key out of notebook variables after client construction.

try:
    _check = GEMINI_CLIENT.interactions.create(
        model=MODEL_NAME,
        input='Reply with exactly: Megumi Gemini brain connected.'
    )
    if not getattr(_check, 'output_text', None):
        raise RuntimeError('Gemini returned an empty response.')
    print('✅ Real Gemini request succeeded.')
    print('Model:', MODEL_NAME)
    print(_check.output_text.strip())
except Exception as exc:
    raise RuntimeError(
        f'Gemini verification failed ({type(exc).__name__}). Check the replacement key, API project, '
        'model availability, quota/billing, and internet connection. The key was not printed.'
    ) from None


In [ ]:
# @title 3. Initialize the second memory, document RAG, and persistent chat journal
from megumi_core import MegumiMemory, MegumiAgent

memory = MegumiMemory(
    persist_dir=MEMORY_DIR,
    memory_collection='megumi_user_memories',
    knowledge_collection='megumi_knowledge_base',
    legacy_collection='megumi_long_term_memory',
    embedding_model='sentence-transformers/all-MiniLM-L6-v2',
)
PERSIST_CHAT_HISTORY = True  # Set False if you prefer not to persist chat transcripts.
agent = MegumiAgent(
    client=GEMINI_CLIENT,
    memory=memory,
    model_name=MODEL_NAME,
    max_history_turns=10,
    history_file=BASE_DIR / 'conversation_history.jsonl',
    persist_history=PERSIST_CHAT_HISTORY,
)
print('✅ Gemini agent + persistent RAG memory initialized.')
print('Memory counts:', memory.counts())
print('Chat history persistence:', PERSIST_CHAT_HISTORY)


## 🧠 4. Second Memory Cell — explicit memory and knowledge ingestion

This cell is kept separate on purpose. The *user memory* collection stores facts you explicitly tell Megumi to remember. The *knowledge base* collection stores uploaded reference documents. Both persist on Drive and are retrieved for relevant questions.

In [ ]:
# @title 4. Second Memory Cell — save facts, recall, ingest files, and inspect memory
from google.colab import files


def remember_for_me(text: str, category: str = 'user-approved'):
    ids = agent.remember(text, category=category)
    print(f'✅ Saved {len(ids)} memory chunk(s).')
    return ids


def recall_from_memory(query: str, top_k: int = 6):
    matches = memory.search(query, k=top_k)
    if not matches:
        print('No matching memory or document was found.')
        return []
    for i, item in enumerate(matches, 1):
        meta = item['metadata']
        print(f"\n{i}. {meta.get('kind', 'reference')} · {meta.get('source', 'unknown')}\n{item['text']}")
    return matches


def upload_and_ingest_knowledge():
    print('Upload PDF, DOCX, TXT, MD, CSV, JSON, or HTML files (maximum 30 MB each).')
    uploaded = files.upload()
    results = []
    for filename in uploaded:
        path = Path('/content') / filename
        try:
            ids = memory.ingest_file(path, source_label=filename)
            results.append({'file': filename, 'chunks': len(ids), 'status': 'indexed'})
        except Exception as exc:
            results.append({'file': filename, 'status': 'error', 'error': f'{type(exc).__name__}: {str(exc)[:200]}'})
    print('Ingestion result:', results)
    return results


def ingest_drive_knowledge_folder():
    result = memory.ingest_folder(KNOWLEDGE_DIR)
    print('Folder ingestion:', result)
    return result


def memory_status():
    result = memory.counts()
    print('Persistent memory:', result, '| total chunks:', sum(result.values()))
    return result

memory_status()
print('Drive knowledge folder:', KNOWLEDGE_DIR)


In [ ]:
# @title 5. Megumi command router and normal Q&A
SEARCH_GROUNDING_ENABLED = True


def route_megumi_command(user_text: str):
    text = (user_text or '').strip()
    low = text.lower()
    if not text:
        return 'Please type a question or command.'
    if low.startswith('/remember '):
        ids = remember_for_me(text[len('/remember '):].strip())
        return f'💾 Saved {len(ids)} memory chunk(s) in the persistent second memory.'
    if low.startswith('/recall '):
        hits = memory.search(text[len('/recall '):].strip(), k=6)
        if not hits:
            return 'No matching memory was found. Add a fact or ingest a document first.'
        return '\n\n'.join(f"[{h['metadata'].get('source','memory')}] {h['text']}" for h in hits)
    if low.startswith('/forget-source '):
        source = text[len('/forget-source '):].strip()
        deleted = memory.forget_source(source)
        return f'Removed {deleted} chunks for exact source: {source}'
    if low == '/memory-status':
        return str(memory.counts())
    if low.startswith('/deep '):
        return agent.coordinated_answer(text[len('/deep '):].strip())['final']
    if low.startswith('/report '):
        result = agent.create_report(text[len('/report '):].strip(), REPORTS_DIR)
        return result['content'] + f"\n\nSaved to Google Drive: {result['path']}"
    if low == '/clear':
        agent.clear_session(clear_persisted_history=True)
        return 'Current and persisted chat history cleared. Saved memories and documents were kept.'
    return agent.ask(text, use_rag=True, top_k=6,
                     use_google_search=None if SEARCH_GROUNDING_ENABLED else False)


def ask_megumi(question: str, deep: bool = False):
    if deep:
        return agent.coordinated_answer(question)['final']
    return route_megumi_command(question)

print('✅ Command router ready: normal Q&A, RAG recall, explicit memory, deep workflow, reports.')


In [ ]:
# @title 6. Test a question (optional live Gemini request)
# Uncomment these lines when you want to send a real question to Gemini.
# print(ask_megumi('Explain how your second memory helps answer my questions.'))
# print(ask_megumi('What should I save in memory for my biotechnology study project?'))
print('Megumi Q&A is ready. Open the avatar interface below or call ask_megumi("your question").')


## 🧩 7. Multi-role agent workflow

For complicated requests Megumi can run a planner → specialist → reviewer sequence. All roles use the same Gemini model, but each stage has a separate objective and is checked before the final answer is returned.

In [ ]:
# @title 7. Run coordinated Gemini agents

def sync_agent_team(question: str, show_intermediate: bool = False):
    result = agent.coordinated_answer(question)
    if show_intermediate:
        print('--- PLANNER ---\n', result['plan'])
        print('\n--- SPECIALIST ---\n', result['draft'])
    print('\n--- REVIEWED ANSWER ---\n', result['final'])
    return result

# Example (uncomment to run):
# sync_agent_team('Create a study plan for hematology and clinical pathology.', show_intermediate=True)


In [ ]:
# @title 8. Generate a report using RAG evidence

def create_megumi_report(topic: str):
    result = agent.create_report(topic, REPORTS_DIR)
    print('✅ Report saved:', result['path'])
    print('Retrieved sources:', result['retrieved_sources'])
    print('\n' + result['content'])
    return result

# Example (uncomment to run):
# create_megumi_report('Western blotting: principle, workflow, controls, applications, and limitations')


In [ ]:
# @title 9. Gemini multilingual voice synthesis
from megumi_voice import synthesize_wav, synthesize_to_file
from IPython.display import Audio, display


def speak_megumi(text: str, voice: str = 'Kore', style: str = 'natural, warm, clear, conversational'):
    output_path = AUDIO_DIR / 'megumi_last_reply.wav'
    saved = synthesize_to_file(GEMINI_CLIENT, text, output_path, voice=voice, style=style)
    display(Audio(saved, autoplay=False))
    return saved

# Optional test (uses a separate TTS API request):
# speak_megumi('வணக்கம்! நான் மெகுமி. இன்று உங்களுக்கு எப்படி உதவலாம்?')


In [ ]:
# @title 10. Full-screen Megumi 3D VRM avatar — only avatar, settings, and bottom keyboard
from megumi_avatar_ui import render_avatar_console

if not AVATAR_PATH.exists():
    raise FileNotFoundError(f'VRM file not found: {AVATAR_PATH}. Re-upload the Master ZIP.')

render_avatar_console(
    agent=agent,
    memory=memory,
    client=GEMINI_CLIENT,
    avatar_path=AVATAR_PATH,
    voice_name='Kore',
    command_handler=route_megumi_command,
)


In [ ]:
# @title 11. Final diagnostics — safe, non-destructive checks
from pathlib import Path
import importlib.util

checks = {
    'Gemini client initialized': GEMINI_CLIENT is not None,
    'RAG memory initialized': memory is not None,
    'Drive memory folder exists': Path(MEMORY_DIR).exists(),
    'Knowledge folder exists': Path(KNOWLEDGE_DIR).exists(),
    'Avatar VRM exists': Path(AVATAR_PATH).exists(),
    'Agent history initialized': isinstance(agent.history, list),
    'Core module importable': importlib.util.find_spec('megumi_core') is not None,
}
for label, passed in checks.items():
    print(('✅' if passed else '❌'), label)
print('Memory counts:', memory.counts())
print('Reports:', REPORTS_DIR)
print('No API key or token is printed by this diagnostic cell.')
